## Plot branchwater output for GTDB only

Used branchwater to look for microbes that are common in pig gut. 
Returns metagenomes. Count for each metagenome how many of these microbes are present. 
If more of these microbes (out of N) are present, is the metagenome likely to be pig? (yes)

In [1]:
# base checkout of workflow directory is here:
BASE='/home/ctbrown/scratch3/2026-pancoding-core/'

BP_THRESHOLD=20000

BASE_OUTPUTS=BASE+'/outputs.branchwater/parquet/'

# where to place figures
FIG_OUT=BASE+'/outputs.figures/'

In [2]:
# imports
import polars as pl
import numpy as np
import matplotlib.pyplot as plt

### Set up human/pig/other labels.

In [3]:
# remove shitty entries with shitty metatada
unnannotated_values = [
    "metagenome",
    "gut metagenome",
    "feces metagenome",
    "manure metagenome",
    "bacterium",
    "unidentified",
    "null"
    ]
def remove_unnannotated(df: pl.DataFrame) -> pl.DataFrame:
    return df.filter(~pl.col("organism").is_in(unnannotated_values))


# group by organism and count number of bacteria per SRA 
def pivot_count(df: pl.DataFrame) -> pl.DataFrame:
    return (
        df
        .group_by(["broad_cat", "count"])
        .len()  # count occurrences
        .pivot(
            values="len",
            index="broad_cat",
            on="count"
        )
        .fill_null(0)  # optional: fill missing with 0
    )

# set mapping for "categories of interest" I like human v pig v neither. 
category_map_simple = {
    'human associated': ['human', 'homo', 'sapiens'],
    'pig': ['pig', 'sus', 'scrofa']}

# Function to assign category based on keywords above
def get_broad_cat_simple(organism):
    for cat, keywords in category_map_simple.items():
        if any(keyword.lower() in str(organism).lower() for keyword in keywords):
            return cat
    return 'other'  

# colormap plotting
colors = {
    'pig': '#264653',
    'human associated': '#e76f51',
    'other': '#f4a261',
    'pig/other': '#2a9d8f',
    'pig-associated': '#e9c46a'
    }

### Load the cds3 manysearch results

In [ ]:
dirpath = BASE_OUTPUTS+'/gtdb.manysearch.*.parquet'

raw_bw_df = (pl.scan_parquet(dirpath).with_columns(
    # remove singlehash from name
    species=pl.col('query_name').str.split(' ').list.slice(1, 2).list.join( " "),
    acc=pl.col("match_name"),
    intersect_bp=pl.col("scaled")*pl.col("intersect_hashes"),
).select(["species", "acc", "containment", "intersect_bp"])
).collect()

n_species = raw_bw_df['species'].n_unique()
n_acc = raw_bw_df['acc'].n_unique()
print(f"loaded {len(raw_bw_df)} branchwater results for {n_species} species containing {n_acc} metagenomes")

### Filter out the metagenomes we used to _find_ the core species

After all, we already know the species are in those!

In [ ]:
names_3216_df = (pl.read_csv('../3216.manifest.csv')
                 .with_columns(acc=pl.col('name'))
                 ['acc'].unique().to_frame()
                )
assert len(names_3216_df) == 3216, len(names_3216_df)
names_3216 = set(names_3216_df['acc'].to_list())

filtered_bw_df = raw_bw_df.filter(~(pl.col('acc').is_in(names_3216)))
n_species = filtered_bw_df['species'].n_unique()
n_acc = filtered_bw_df['acc'].n_unique()

print(f"remaining after removing 3216 metag: {len(filtered_bw_df)} branchwater results for {n_species} species containing {n_acc} metagenomes")

### Load metadata for datasets & get rid of non-WGS datasets

In [ ]:
# get metadata, only compare WGS
metadata_df = (
    pl.scan_parquet("/group/ctbrowngrp5/sra-metagenomes/20241128-metadata.parquet")
    .filter(pl.col("acc") != "NP")
    .filter(pl.col("assay_type") == "WGS")
#    .select(["acc", "organism", "bioproject", "mbases", "host", "project_name"]) 
    .collect()
)
print(f"found metadata records for {len(metadata_df)} WGS metagenome data sets from the SRA")

In [ ]:
big_names_df = pl.read_csv('../inputs.branchwater/wort-acc-gt-1e6.csv')

### Create a new dataframe that will provide a 'simpleorg' column: human, pig, other

In [ ]:
rename_df = []
for organism in metadata_df['organism'].unique().to_list():
    simpleorg = 'other'
    if organism:
        for kw in ['human', 'homo']:
            if kw in organism.lower():
                simpleorg = 'human'
                break
        for kw in ['pig', 'sus', 'scrofa']:
            if kw in organism.lower():
                simpleorg = 'pig'
                break
    rename_df.append(dict(organism=organism, broad_cat=simpleorg))

rename_df = pl.DataFrame(rename_df)
rename_df['broad_cat'].value_counts().sort(by='count', descending=True)

### Apply the renaming to the metadata table => metadata2_df

In [ ]:
metadata2_df = metadata_df.join(rename_df, on='organism', how='inner', coalesce=True)
assert len(metadata2_df) == len(metadata_df)
metadata2_df

In [ ]:
n_pig = len(metadata2_df.filter(pl.col('broad_cat') == 'pig'))
n_human = len(metadata2_df.filter(pl.col('broad_cat') == 'human'))

print(f"{n_pig} pig, {n_human} human metag")

### Bring in the metadata to the branchwater dataframe

In [ ]:
bw_metadata_df = filtered_bw_df.join(metadata2_df, on="acc", how="inner")
print(f"{bw_metadata_df['species'].n_unique()} species, {bw_metadata_df['acc'].n_unique()} metagenomes")

In [ ]:
bw_remove_df = bw_metadata_df.join(big_names_df, left_on='acc', right_on='name', how='anti')
print(len(bw_metadata_df), len(bw_remove_df))
print(f"{bw_remove_df['species'].n_unique()} species, {bw_remove_df['acc'].n_unique()} metagenomes")

### Create a core_bw_df that has just the core species records in it 

In [ ]:
CORE_SPECIES=set([ x.strip() for x in open('../inputs.branchwater/names.list') ])
core_bw_df = bw_remove_df.filter(pl.col("species").is_in(CORE_SPECIES))
print(f"{core_bw_df['species'].n_unique()} species left after filtering for core; {core_bw_df['acc'].n_unique()} metagenomes")
assert core_bw_df['species'].n_unique() == len(CORE_SPECIES)

## Apply thresholds and remove entries with bad metadata

In [ ]:
# apply hash threshold
bw_df = core_bw_df.filter(pl.col("intersect_bp") >= BP_THRESHOLD)

# remove unnanotated metags (metagenome, gut metagenome, etc)
df = remove_unnannotated(bw_df)

# update with organism broad category
df = df.join(rename_df, on='organism', how='inner')

print(f'Removed unnanotated & added simpleorg: {len(bw_df)} -> {len(df)} (-{len(bw_df) - len(df)})')

# count number of orgs (out of 16) in each mG
counts_df = (
    df
    .group_by(["acc", "broad_cat"]).agg(
        pl.len().alias("count")
    )
)

print(counts_df)

# pivot for plotting
pl_dfp = pivot_count(counts_df)

# sort columns by number
assert pl_dfp.columns[0] == 'broad_cat'
sorted_cols = list(sorted(pl_dfp.columns[1:], key=lambda x: int(x)))
pl_dfp = pl_dfp.select(['broad_cat'] + sorted_cols)

print(pl_dfp)

In [ ]:
# transpose with sensible column names
df_cm = pl_dfp.transpose(include_header=True, header_name='index',
                 column_names=pl_dfp['broad_cat'].to_list())

# get rid of first row
df_cm = df_cm.tail(-1)

# change the values to all integer
df_cm = df_cm.with_columns(
    pl.col("index").str.to_integer(),
    pl.col("pig").str.to_integer(),
    pl.col("human").str.to_integer(),
    pl.col("other").str.to_integer()
)

df_cm = df_cm.with_columns(
    pl.col("pig").cum_sum(reverse=True).alias("cm_pig"),
    pl.col("human").cum_sum(reverse=True).alias("cm_human"),
    pl.col("other").cum_sum(reverse=True).alias("cm_other"),
)


df_cm = df_cm.with_columns(
    (pl.col("cm_pig") / (pl.col("cm_pig") + pl.col("cm_human") + pl.col("cm_other"))).alias("f_pig"),
    (pl.col("cm_human") / (pl.col("cm_pig") + pl.col("cm_human") + pl.col("cm_other"))).alias("f_human"),
    (pl.col("cm_other") / (pl.col("cm_pig") + pl.col("cm_human") + pl.col("cm_other"))).alias("f_other"),
)

(df_cm
 .select(['index', 'cm_pig', 'cm_human', 'cm_other', 'f_pig', 'f_human', 'f_other'])
 .style
 .fmt_number(decimals=3, columns=['f_pig', 'f_human', 'f_other'])
)

In [ ]:
xlabels = df_cm['index'].to_list()
weights = {}
for col in ('cm_pig', 'cm_human', 'cm_other', 'f_pig', 'f_human', 'f_other'):
    weights[col] = df_cm[col].to_list()

labels = {}
labels['cm_human'] = 'human-associated'
labels['cm_other'] = 'other'
labels['cm_pig'] = 'pig-associated'
labels['f_human'] = 'human-associated'
labels['f_other'] = 'other'
labels['f_pig'] = 'pig-associated'

# colormap plotting
colors = {
    'cm_pig': '#264653',
    'cm_human': '#e76f51',
    'cm_other': '#f4a261',
    'f_pig': '#264653',
    'f_human': '#e76f51',
    'f_other': '#f4a261',
    }

fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(8, 4))

bottom = np.zeros(len(xlabels))
for name in 'cm_pig', 'cm_human', 'cm_other':
    counts = weights[name]
    axes[0].bar(xlabels, counts, label=labels[name], bottom=bottom, color=colors[name])
    bottom += np.array(counts)

bottom = np.zeros(len(xlabels))
for name in 'f_pig', 'f_human', 'f_other':
    counts = weights[name] 
    axes[1].bar(xlabels, counts, label=labels[name], bottom=bottom, color=colors[name])
    bottom += np.array(counts)

for ax in axes:
    ax.set_xlim(0.3, 16.7)
    ax.set_xlabel('Number of bacterial species (out of 16)')
    ax.tick_params(axis='x', labelrotation=90)
    ax.set_xticks(xlabels)

axes[0].legend(title='Metagenome label', loc='upper right')
axes[0].set_ylabel('Number of metagenomes')
axes[1].set_ylabel('Fraction of metagenomes')

plt.savefig('foo.png')

## Figure 1: Core bacteria largely select against human-associated metagenomes.

21-mer search of 700,000 public metagenomes with each of 16 bacterial species pangenomes  found almost 200,000 metagenomes containing at least one species at a containment threshold of 5%. Over 163,000 of these metagenomes were human-associated. Filtering metagenomes based on the number of contained species progressively eliminated the large majority of human metagenomes; when requiring 12 or more species, 80% of the pig metagenomes were retained while only 0.2% of the human metagenomes were retained.

## Calculate enrichment rates

In [ ]:
d = metadata2_df['broad_cat'].value_counts().to_dict(as_series=False)

pig_idx = d['broad_cat'].index('pig')
n_orig_pig = d['count'][pig_idx]

human_idx = d['broad_cat'].index('human')
n_orig_human = d['count'][human_idx]

f_orig_pig = n_orig_pig / (n_orig_pig + n_orig_human)
f_orig_human = n_orig_human / (n_orig_pig + n_orig_human)

orig_pig_ratio = n_orig_pig / (n_orig_pig + n_orig_human)
orig_human_ratio = n_orig_human / (n_orig_pig + n_orig_human)

print(orig_pig_ratio, orig_human_ratio)

In [ ]:

enrich_df = df_cm.with_columns(
    enrich_pig = (pl.col("cm_pig") / (pl.col("cm_pig") + pl.col("cm_human")) / orig_pig_ratio),
    enrich_human = (pl.col("cm_human") / (pl.col("cm_pig") + pl.col("cm_human")) / orig_human_ratio),
)

(enrich_df
 .select(['index', 'cm_human', 'cm_pig', 'enrich_pig', 'enrich_human'])
 .style
 .fmt_number(columns=['enrich_human', 'enrich_pig'], decimals=2)
)

In [ ]:
# plt.plot(enrich_df['index'], enrich_df['enrich_human'], label='human enrichment')
plt.plot(enrich_df['index'], enrich_df['enrich_pig'], label='pig enrichment')
plt.title('enrichment for pig microbiome data sets over human')
plt.xlabel('number of pig core species found (of 16)')
plt.ylabel('enrichment')
plt.xticks(range(1, 17))

#plt.legend(loc='upper right')
plt.ylim(0, 40)